# SQL with pandas

## Introduction

pandas and SQL are complementary tools — SQL is great for querying large relational databases, and pandas is great for analysis and visualisation once you have the data. This notebook covers three ways to combine them:

1. `pd.read_sql()` — execute a SQL query directly into a DataFrame
2. `DataFrame.query()` — filter a DataFrame with a SQL-like string
3. `pandasql` — write full SQL queries against in-memory DataFrames

## Objectives

You will be able to:

- Use `pd.read_sql()` to pull query results from a SQLite database into a DataFrame
- Filter DataFrames with the `.query()` method and understand how it differs from boolean indexing
- Write SQL queries against DataFrames using `pandasql`

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

---

## `pd.read_sql()` — From a Database to a DataFrame

`pd.read_sql(sql, con)` executes a SQL query and returns the result as a DataFrame with column names set automatically. It replaces the `cur.execute()` + `pd.DataFrame()` + `df.columns` pattern from earlier notebooks.

```python
conn = sqlite3.connect('path/to/database.sqlite')
df = pd.read_sql('SELECT * FROM table LIMIT 10;', conn)
```

In [ ]:
conn = sqlite3.connect('data/selecting_data/data.sqlite')

# Clean one-liner — no cursor, no manual column assignment
employees = pd.read_sql('SELECT * FROM employees LIMIT 5;', conn)
employees

In [ ]:
# Multi-line queries work just as well
top_customers = pd.read_sql("""
    SELECT customerName, city, creditLimit
    FROM customers
    ORDER BY creditLimit DESC
    LIMIT 10;
""", conn)
top_customers

In [ ]:
# Aggregation and GROUP BY — full SQL syntax supported
revenue_by_country = pd.read_sql("""
    SELECT country,
           COUNT(customerNumber) AS num_customers,
           AVG(creditLimit) AS avg_credit
    FROM customers
    GROUP BY country
    ORDER BY num_customers DESC
    LIMIT 10;
""", conn)
revenue_by_country

---

## `DataFrame.query()` — SQL-like Filtering in pandas

`.query()` lets you filter a DataFrame by writing the condition as a string — similar to a SQL `WHERE` clause. It's often more readable than boolean indexing for compound conditions.

```python
# Boolean indexing
df[(df['Age'] < 15) & (df['Sex'] == 'female')]

# Equivalent with .query()
df.query('Age < 15 and Sex == "female"')
```

In [ ]:
# Load the Titanic dataset
titanic = pd.read_csv('data/using_sql_with_pandas_lab/titanic.csv')
print(titanic.shape)
titanic.head(3)

In [ ]:
# Boolean indexing — 2nd or 3rd class passengers
non_first = titanic[titanic['Pclass'] >= 2]
print(f"Non-first class: {len(non_first)}")

# .query() equivalent
non_first_q = titanic.query('Pclass >= 2')
print(f"Non-first class (query): {len(non_first_q)}")

In [ ]:
# Compound condition — fares between 50 and 100
mid_fares = titanic.query('50 <= Fare <= 100')
print(f"Passengers with fare 50–100: {len(mid_fares)}")
mid_fares['Fare'].hist()
plt.xlabel('Fare')
plt.title('Distribution of Mid-Range Fares')
plt.tight_layout()
plt.show()

In [ ]:
# String comparison in .query() — note the inner quotes
female_children = titanic.query('Sex == "female" and Age <= 15')
print(f"Female passengers aged ≤15: {len(female_children)}")
female_children.head()

In [ ]:
# Survival comparison: women & children vs adult men
women_children = titanic.query('Sex == "female" or Age <= 15')
adult_men = titanic.query('Sex == "male" and Age > 15')

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
women_children['Survived'].hist(ax=axes[0])
axes[0].set_title('Women & Children Survived')
axes[0].set_xlabel('0=Died, 1=Survived')

adult_men['Survived'].hist(ax=axes[1])
axes[1].set_title('Adult Men Survived')
axes[1].set_xlabel('0=Died, 1=Survived')

plt.tight_layout()
plt.show()

---

## `pandasql` — SQL Queries Against DataFrames

`pandasql` lets you run full SQL queries against pandas DataFrames as if they were database tables. The table name in your SQL is the **variable name** of the DataFrame in the current scope.

```python
from pandasql import sqldf
pysqldf = lambda q: sqldf(q, globals())

result = pysqldf("SELECT * FROM titanic WHERE Age < 15 LIMIT 5;")
```

In [ ]:
# Install pandasql if needed: pip install pandasql
from pandasql import sqldf

pysqldf = lambda q: sqldf(q, globals())

In [ ]:
# Basic SELECT with LIMIT — table name is the DataFrame variable name 'titanic'
pysqldf("SELECT Name, Sex, Age, Survived FROM titanic LIMIT 10;")

In [ ]:
# Surviving male passengers with fare details
pysqldf("""
    SELECT Name, Fare, Survived
    FROM titanic
    WHERE Sex = 'male' AND Survived = 1
    ORDER BY Fare DESC
    LIMIT 10;
""")

In [ ]:
# GROUP BY — survival rate and passenger count per class
pysqldf("""
    SELECT Pclass,
           COUNT(*) AS passengers,
           AVG(Survived) AS survival_rate
    FROM titanic
    GROUP BY Pclass
    ORDER BY Pclass;
""")

In [ ]:
# Survived females by Pclass — visualised
survived_f = pysqldf("SELECT Pclass FROM titanic WHERE Sex = 'female' AND Survived = 1;")
died_f = pysqldf("SELECT Pclass FROM titanic WHERE Sex = 'female' AND Survived = 0;")

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
survived_f['Pclass'].value_counts().sort_index().plot(kind='barh', ax=axes[0])
axes[0].set_title('Female survivors by class')
died_f['Pclass'].value_counts().sort_index().plot(kind='barh', ax=axes[1])
axes[1].set_title('Female fatalities by class')
plt.tight_layout()
plt.show()

---

## Practice

Use whichever method you prefer (`boolean indexing`, `.query()`, or `pandasql`) to answer the following.

In [ ]:
# 1. Use pd.read_sql() to pull all customers from the USA from the CRM database


In [ ]:
# 2. Using .query(), get all Titanic passengers with a Fare > 100 who survived


In [ ]:
# 3. Using pandasql, compute the average Age and average Fare by Sex and Pclass


In [ ]:
# 4. Using pandasql, find the survival rate for passengers whose Fare was between
#    0 and 50, 50 and 100, and above 100 — use a CASE WHEN or just three queries


---

## Summary

In this notebook you learned three complementary ways to combine SQL and pandas:

- **`pd.read_sql(query, conn)`** — the cleanest way to pull database query results into a DataFrame without a cursor or manual column assignment
- **`df.query(string)`** — a readable SQL-like syntax for filtering DataFrames, supporting `and`/`or` and range conditions like `50 <= col <= 100`
- **`pandasql.sqldf()`** — full SQL (SELECT, WHERE, GROUP BY, ORDER BY) against in-memory DataFrames, treating the variable name as the table name